<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/14_document_chunking_rag/document_chunking_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
document = """
Paris is the capital city of France. It is one of the most visited cities in the world.
France is located in Europe and has a rich cultural history.
The Eiffel Tower is a famous landmark in Paris.
Millions of tourists visit Paris every year.
"""

In [13]:
def chunk_text(text, chunk_size=20):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

chunks = chunk_text(document)

print("DOCUMENT CHUNKS:\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}\n")

DOCUMENT CHUNKS:

Chunk 1: Paris is the capital city of France. It is one of the most visited cities in the world. France is

Chunk 2: located in Europe and has a rich cultural history. The Eiffel Tower is a famous landmark in Paris. Millions of

Chunk 3: tourists visit Paris every year.



In [14]:
import faiss
import numpy as np

chunk_embeddings = embed_model.encode(chunks)

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

In [15]:
query = "What is the capital of France?"

query_embedding = embed_model.encode([query])

In [16]:
D, I = index.search(np.array(query_embedding), k=1)

retrieved_chunk = chunks[I[0][0]]

print("RETRIEVED CHUNK:")
print(retrieved_chunk)

RETRIEVED CHUNK:
Paris is the capital city of France. It is one of the most visited cities in the world. France is


In [17]:
prompt = f"""
Use the following context to answer the question.

Context: {retrieved_chunk}

Question: {query}
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
Paris
